In [ ]:
"""
آزمون فرض نسبت دو نمونه‌ای
آسانسور و بازسازی
"""

import pandas as pd
from statsmodels.stats.proportion import proportions_ztest


file_path = "final_data.csv"

df = pd.read_csv(
    file_path,
    usecols=["has_elevator", "is_rebuilt"])

print("Initial rows:", len(df))


df = df.dropna(
    subset=["has_elevator", "is_rebuilt"])

print("Rows after removing missing values:", len(df))


# تعداد موفقیت‌ها در هر گروه
elevator_success = len(
    df[
        (df["has_elevator"] == 1) &
        (df["is_rebuilt"] == 1)
    ])

no_elevator_success = len(
    df[
        (df["has_elevator"] == 0) &
        (df["is_rebuilt"] == 1)
    ])


# تعداد کل نمونه‌ها در هر گروه
n_elevator = len(
    df[df["has_elevator"] == 1])

n_no_elevator = len(
    df[df["has_elevator"] == 0])


# نسبت بازسازی در هر گروه
p_elevator = elevator_success / n_elevator
p_no_elevator = no_elevator_success / n_no_elevator


print("\nGroup information:")
print(
    f"Elevator: n={n_elevator}, "
    f"success={elevator_success}, "
    f"p={p_elevator:.3f}")

print(
    f"No Elevator: n={n_no_elevator}, "
    f"success={no_elevator_success}, "
    f"p={p_no_elevator:.3f}")


# بررسی شرط نرمال بودن تقریبی
print("\nChecking conditions for normal approximation:")

print(
    f"Elevator: n*p = {n_elevator * p_elevator:.1f}, "
    f"n*(1-p) = {n_elevator * (1-p_elevator):.1f}"
)

print(
    f"No Elevator: n*p = {n_no_elevator * p_no_elevator:.1f}, "
    f"n*(1-p) = {n_no_elevator * (1-p_no_elevator):.1f}"
)


if (
    n_elevator * p_elevator >= 5 and
    n_elevator * (1-p_elevator) >= 5 and
    n_no_elevator * p_no_elevator >= 5 and
    n_no_elevator * (1-p_no_elevator) >= 5
):
    print("\nConditions are met.")

    # Two-Sample Proportion Z-Test
    z_stat, p_value = proportions_ztest(
        count=[elevator_success, no_elevator_success],
        nobs=[n_elevator, n_no_elevator],
        alternative="two-sided"
    )

    print("\nTwo-Sample Proportion Z-Test")
    print("H0: p_elevator = p_no_elevator")
    print("H1: p_elevator != p_no_elevator")
    print("z-statistic:", z_stat)
    print("p-value:", p_value)

    alpha = 0.05

    if p_value < alpha:
        print("\nReject H0")
        print("There is a significant difference between the two proportions.")
    else:
        print("\nFail to reject H0")
        print("There is no significant difference between the two proportions.")

else:
    print("\nConditions are not met.")

Initial rows: 1000000
Rows after removing missing values: 381932

Contingency table:
is_rebuilt     False  True 
has_elevator               
False          77996  59420
True          163144  81372

Chi-Square test
Chi-square statistic: 3750.9449096114013
p-value: 0.0
Degrees of freedom: 1

Reject H0
There is a significant relationship between the variables.


In [ ]:
"""
آزمون فرض امتیازی دوم 
Kruskal-Wallis
جهت ساختمان و قیمت
"""
import pandas as pd
from scipy.stats import kruskal


file_path = "final_data.csv"

df = pd.read_csv(
    file_path,
    usecols=["price_value", "building_direction"])

print("Initial rows:", len(df))


df = df.dropna(
    subset=["price_value", "building_direction"])

df["price_value"] = pd.to_numeric(
    df["price_value"],
    errors="coerce")

df = df.dropna(subset=["price_value"])

print("Rows after removing missing values:", len(df))


# محدود کردن داده های پرت با IQR

Q1 = df["price_value"].quantile(0.25)
Q3 = df["price_value"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# جایگزین کردن مقادیر پرت با حد مجاز IQR
df["price_value"] = df["price_value"].clip(
    lower=lower,
    upper=upper)

print("Rows after handling outliers:", len(df))

print("\nBuilding direction:")
print(df["building_direction"].value_counts())


groups = []

for direction in df["building_direction"].unique():

    group = df.loc[
        df["building_direction"] == direction,
        "price_value"]

    groups.append(group)


statistic, p_value = kruskal(*groups)


print("\nKruskal-Wallis test")
print("H statistic:", statistic)
print("p-value:", p_value)


alpha = 0.05

if p_value < alpha:

    print("\nReject H0")
    print("There is a significant difference between the groups.")

else:

    print("\nFail to reject H0")
    print("There is no significant difference between the groups.")


# مقایسه قیمت ملک بر اساس جهت ساختمان

summary = df.groupby("building_direction")["price_value"].agg(
    count="count",
    mean="mean",
    median="median",
    min="min",
    max="max"
).sort_values("median", ascending=False)

print("\nPrice comparison by building direction:")
print(summary)

Initial rows: 1000000
Rows after removing missing values: 202871
Rows after handling outliers: 202871

Building direction:
building_direction
south    100670
north     90073
east       7674
west       4454
Name: count, dtype: int64

Kruskal-Wallis test
H statistic: 2829.6358079302295
p-value: 0.0

Reject H0
There is a significant difference between the groups.

Price comparison by building direction:
                     count          mean        median  min           max
building_direction                                                       
south               100670  5.199437e+09  3.800000e+09  0.0  1.328750e+10
north                90073  4.480585e+09  3.200000e+09  0.0  1.328750e+10
east                  7674  3.874512e+09  2.800000e+09  0.0  1.328750e+10
west                  4454  3.868911e+09  2.780000e+09  0.0  1.328750e+10
